<a href="https://colab.research.google.com/github/syoon7079/tjwn/blob/main/10_table1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 10주차 Table 1 — 외식 빈도와 BMI (6팀)
- 자료: 국민건강영양조사 제9기 3차년도(2024) `HN24_ALL.sav` (Drive `MyDrive/hdac_data/`)
- 노출: 외식 빈도 `L_OUT_FQ` (저·중·고빈도) / 결과: `HE_BMI` / 설계: `kstrata`, `psu`, `wt_tot`
- 실행: **런타임 → 세션 다시 시작 및 모두 실행** → 'GitHub에 사본 저장'
- 마지막 셀이 `docs/10_table1.md` 내용을 **노트북 출력 숫자 그대로** 만들어 저장합니다. 개인별 행은 출력하지 않습니다.

In [1]:
# 0) 설정
from google.colab import drive
drive.mount('/content/drive')
import os, numpy as np, pandas as pd
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 60); pd.set_option('display.max_rows', 200)

DATA_DIR = '/content/drive/MyDrive/hdac_data'            # 팀 공통 경로
SAV_PATH = f'{DATA_DIR}/HN24_ALL.sav'
OUT_DIR  = f'{DATA_DIR}/output'                           # 집계표(md)만 저장
os.makedirs(OUT_DIR, exist_ok=True)

AGE_MIN  = 19                  # 연구대상: 19세 이상 성인
WEIGHT   = 'wt_tot'            # 건강설문 AND 검진 AND 영양 결합 가중치 (이용지침서 표 6-2 p.28, 변수 설명 p.94)
KEEP_NA8 = True                # BD1_11, BS3_1 의 8(비해당)을 별도 범주로 유지 (False면 결측 처리)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
# 1) 변수 정의표의 변수만 불러오기
!pip -q install pyreadstat
import pyreadstat
NEEDED = ['year', 'l_out_fq', 'he_bmi', 'bd1_11', 'region', 'sex', 'age', 'educ', 'incm', 'bs3_1',
          'kstrata', 'psu', WEIGHT]
_, m0 = pyreadstat.read_sav(SAV_PATH, metadataonly=True)
low2real = {c.lower(): c for c in m0.column_names}
absent = [c for c in NEEDED if c not in low2real]
assert not absent, f'원자료에 없는 변수: {absent}'
df, meta = pyreadstat.read_sav(SAV_PATH, usecols=[low2real[c] for c in NEEDED])
df.columns = df.columns.str.lower()
print(f'불러온 자료: {len(df):,}행 x {df.shape[1]}열 | 조사연도:', sorted(df['year'].dropna().unique()))

불러온 자료: 6,997행 x 13열 | 조사연도: [np.float64(2024.0)]


In [3]:
# 2) 노출·결과·공변량 만들기 (행은 지우지 않고 새 열만 추가)
def num(s): return pd.to_numeric(s, errors='coerce')

def check_recode(raw, new, title):
    # 원 코드 빈도표: 원 코드 × 새 범주 (집계표)
    r = raw.map(lambda v: 'NaN' if pd.isna(v) else str(int(v)) if float(v).is_integer() else str(v))
    print(f'\n--- {title} ---')
    display(pd.crosstab(r, new.astype('object').fillna('(결측)')))

# 2-1. 노출: 외식 빈도 3범주
EXP_VALID = [1, 2, 3, 4, 5, 6, 7]
GRP_MAP   = {5: 1, 6: 1, 7: 1,        # 주 1~2회, 월 1~3회, 거의 안 함 -> 저빈도 (비교군)
             4: 2,                    # 주 3~4회 -> 중빈도
             1: 3, 2: 3, 3: 3}        # 하루 2회+, 하루 1회, 주 5~6회 -> 고빈도 (노출군)
assert sorted(GRP_MAP) == EXP_VALID, '원 코드 누락/중복'
GRP_LABEL = {1: '저빈도 외식군', 2: '중빈도 외식군', 3: '고빈도 외식군'}
df['exp_raw']   = num(df['l_out_fq'])
df['exp_valid'] = df['exp_raw'].isin(EXP_VALID)
df['grp']       = df['exp_raw'].where(df['exp_valid']).map(GRP_MAP)
check_recode(df['exp_raw'], df['grp'].map(GRP_LABEL), 'L_OUT_FQ 원 코드 → 3범주')

# 2-2. 결과, 연령, 가중치
df['bmi']     = num(df['he_bmi'])
df['age_n']   = num(df['age'])
df['w']       = num(df[WEIGHT])
df['obese25'] = np.where(df['bmi'].notna(), (df['bmi'] >= 25).astype(float), np.nan)   # 비만: BMI >= 25
print('\nHE_BMI 요약:'); print(df['bmi'].describe().round(2).to_string())
print('연령 요약:');   print(df['age_n'].describe().round(2).to_string())

# 2-3. 범주형 공변량
def recode(var, mapping, bad=()):
    s = num(df[var]).mask(num(df[var]).isin(bad))      # 모름·무응답 등 -> NaN
    unexpected = sorted(set(s.dropna().unique()) - set(mapping))
    if unexpected: print(f'[주의] {var}: 정의표에 없는 코드 {unexpected} -> 결측 처리됨')
    return s.map(mapping)

na8 = () if KEEP_NA8 else (8,)
CAT_SPECS = [   # (표시명, 원변수, {코드: 라벨}, 결측 처리할 코드)
    ('성별', 'sex', {1: '남성', 2: '여성'}, (9, 99)),
    ('지역', 'region', {1: '서울', 2: '부산', 3: '대구', 4: '인천', 5: '광주', 6: '대전', 7: '울산', 8: '세종',
                       9: '경기', 10: '강원', 11: '충북', 12: '충남', 13: '전북', 14: '전남', 15: '경북',
                       16: '경남', 17: '제주'}, (99,)),
    ('교육수준', 'educ', {1: '초졸 이하', 2: '초졸 이하', 3: '초졸 이하', 4: '중졸', 5: '고졸',
                        6: '대졸 이상', 7: '대졸 이상', 8: '대졸 이상'}, (88, 99)),
    ('소득 4분위(개인)', 'incm', {1: '하', 2: '중하', 3: '중상', 4: '상'}, (8, 9, 99)),
    ('음주 빈도(최근 1년)', 'bd1_11', {1: '전혀 마시지 않음', 2: '월 1회 미만', 3: '월 1회 정도', 4: '월 2-4회',
                                   5: '주 2-3회', 6: '주 4회 이상', 8: '비해당(코드 8)'}, (9, 99) + na8),
    ('현재 흡연', 'bs3_1', {1: '매일 흡연', 2: '가끔 흡연', 3: '과거 흡연(현재 비흡연)',
                          8: '비해당(코드 8)'}, (9, 99) + na8),
]
for label, var, mapping, bad in CAT_SPECS:
    df['c_' + var] = recode(var, mapping, bad)
    check_recode(num(df[var]), df['c_' + var], f'{label} ({var}) 원 코드 → 범주')

# 2-4. 설계변수 유효 여부
psu_str = df['psu'].astype(str).str.strip()
df['psu_ok']    = df['psu'].notna() & ~psu_str.isin(['', 'nan', 'None'])
df['design_ok'] = df['w'].notna() & (df['w'] > 0) & df['kstrata'].notna() & df['psu_ok']

# 2-5. 분석 대상 표시변수 (행 삭제 없음) — 공변량 결측은 여기에 넣지 않는다
df['s1'] = True
df['s2'] = df['age_n'] >= AGE_MIN
df['s3'] = df['s2'] & df['exp_valid']
df['s4'] = df['s3'] & df['bmi'].notna()
df['s5'] = df['s4'] & df['design_ok']
df['analytic'] = df['s5']
print(f"\n최종 분석 대상(analytic=True): {int(df['analytic'].sum()):,}명 / 전체 {len(df):,}행 유지")


--- L_OUT_FQ 원 코드 → 3범주 ---


grp,(결측),고빈도 외식군,저빈도 외식군,중빈도 외식군
exp_raw,,,,
1,0,246,0,0
2,0,1083,0,0
3,0,1477,0,0
4,0,0,0,829
5,0,0,1457,0
6,0,0,1199,0
7,0,0,514,0
NaN,192,0,0,0



HE_BMI 요약:
count    6881.00
mean       23.43
std         4.31
min        11.56
25%        20.62
50%        23.27
75%        26.00
max        48.68
연령 요약:
count    6997.00
mean       48.10
std        22.07
min         1.00
25%        31.00
50%        52.00
75%        66.00
max        80.00

--- 성별 (sex) 원 코드 → 범주 ---


c_sex,남성,여성
sex,,
1,3070,0
2,0,3927



--- 지역 (region) 원 코드 → 범주 ---


c_region,강원,경기,경남,경북,광주,대구,대전,부산,서울,세종,울산,인천,전남,전북,제주,충남,충북
region,,,,,,,,,,,,,,,,,
1,0,0,0,0,0,0,0,0,1276,0,0,0,0,0,0,0,0
10,220,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
11,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,213
12,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,300,0
13,0,0,0,0,0,0,0,0,0,0,0,0,0,250,0,0,0
14,0,0,0,0,0,0,0,0,0,0,0,0,231,0,0,0,0
15,0,0,0,364,0,0,0,0,0,0,0,0,0,0,0,0,0
16,0,0,435,0,0,0,0,0,0,0,0,0,0,0,0,0,0
17,0,0,0,0,0,0,0,0,0,0,0,0,0,0,145,0,0


... (5 outputs left out)

c_bd1_11,(결측),비해당(코드 8),월 1회 미만,월 1회 정도,월 2-4회,전혀 마시지 않음,주 2-3회,주 4회 이상
bd1_11,,,,,,,,
1,0,0,0,0,0,1115,0,0
2,0,0,1179,0,0,0,0,0
3,0,0,0,671,0,0,0,0
4,0,0,0,0,1219,0,0,0
5,0,0,0,0,0,0,800,0
6,0,0,0,0,0,0,0,292
8,0,1496,0,0,0,0,0,0
9,216,0,0,0,0,0,0,0
NaN,9,0,0,0,0,0,0,0



--- 현재 흡연 (bs3_1) 원 코드 → 범주 ---


c_bs3_1,(결측),가끔 흡연,과거 흡연(현재 비흡연),매일 흡연,비해당(코드 8)
bs3_1,,,,,
1,0,0,0,681,0
2,0,118,0,0,0
3,0,0,1368,0,0
8,0,0,0,0,4605
9,216,0,0,0,0
NaN,9,0,0,0,0



최종 분석 대상(analytic=True): 5,765명 / 전체 6,997행 유지


In [4]:
# 3) B-1 흐름도: 단계 / 기준 / 남은 인원 / 제외 인원과 이유
def to_md(d):
    cols = list(d.columns)
    lines = ['| ' + ' | '.join(map(str, cols)) + ' |', '|' + '|'.join(['---'] * len(cols)) + '|']
    for _, r in d.iterrows(): lines.append('| ' + ' | '.join(str(x) for x in r) + ' |')
    return '\n'.join(lines)

def reason_text(labels):
    vc = pd.Series(list(labels)).value_counts()
    return '; '.join(f'{k} {v:,}명' for k, v in vc.items()) if len(vc) else '-'

flow = [['1', '2024 원자료 전체 (행 삭제 없음)', f"{int(df['s1'].sum()):,}", '-', '-']]

sub = df[df['s1'] & ~df['s2']]
lab = np.select([sub['age_n'].isna()], ['연령 결측'], default=f'비해당(만 {AGE_MIN}세 미만)')
flow.append(['2', f'만 {AGE_MIN}세 이상', f"{int(df['s2'].sum()):,}", f'{len(sub):,}', reason_text(lab)])

sub = df[df['s2'] & ~df['s3']]
lab = np.select([sub['exp_raw'].isna(), sub['exp_raw'].eq(8), sub['exp_raw'].isin([9, 99, 888])],
                ['결측(NaN)', '비해당(코드 8)', '모름·무응답(코드 9 등)'], default='기타 무효 코드')
flow.append(['3', 'L_OUT_FQ 유효(코드 1~7)', f"{int(df['s3'].sum()):,}", f'{len(sub):,}', reason_text(lab)])

sub = df[df['s3'] & ~df['s4']]
flow.append(['4', 'HE_BMI 유효', f"{int(df['s4'].sum()):,}", f'{len(sub):,}',
             reason_text(['결측(NaN: 검진 미참여 또는 BMI 미산출)'] * len(sub))])

sub = df[df['s4'] & ~df['s5']]
lab = np.select([sub['w'].isna(), sub['w'].eq(0), sub['kstrata'].isna(), ~sub['psu_ok']],
                [f'설계상 제외({WEIGHT} 결측)', f'설계상 제외({WEIGHT} = 0)', '설계상 제외(kstrata 결측)',
                 '설계상 제외(psu 결측)'], default='기타')
flow.append(['5', f'{WEIGHT}(>0), kstrata, psu 유효', f"{int(df['s5'].sum()):,}", f'{len(sub):,}', reason_text(lab)])
flow.append(['6', '최종 분석 대상 (analytic=True, 공변량 결측은 제외하지 않음)', f"{int(df['analytic'].sum()):,}", '-', '-'])

NG = {g: int(((df['grp'] == g) & df['analytic']).sum()) for g in (1, 2, 3)}
flow.append(['6-1', '비교군: 저빈도 외식군 (L_OUT_FQ 5·6·7)', f'{NG[1]:,}', '-', '-'])
flow.append(['6-2', '중빈도 외식군 (L_OUT_FQ 4)', f'{NG[2]:,}', '-', '-'])
flow.append(['6-3', '노출군: 고빈도 외식군 (L_OUT_FQ 1·2·3)', f'{NG[3]:,}', '-', '-'])
flow_df = pd.DataFrame(flow, columns=['단계', '기준', '남은 인원(명)', '제외 인원(명)', '제외 사유'])
FLOW_MD = to_md(flow_df)
print(FLOW_MD)
assert NG[1] + NG[2] + NG[3] == int(df['analytic'].sum())

| 단계 | 기준 | 남은 인원(명) | 제외 인원(명) | 제외 사유 |
|---|---|---|---|---|
| 1 | 2024 원자료 전체 (행 삭제 없음) | 6,997 | - | - |
| 2 | 만 19세 이상 | 6,033 | 964 | 비해당(만 19세 미만) 964명 |
| 3 | L_OUT_FQ 유효(코드 1~7) | 5,859 | 174 | 결측(NaN) 174명 |
| 4 | HE_BMI 유효 | 5,765 | 94 | 결측(NaN: 검진 미참여 또는 BMI 미산출) 94명 |
| 5 | wt_tot(>0), kstrata, psu 유효 | 5,765 | 0 | - |
| 6 | 최종 분석 대상 (analytic=True, 공변량 결측은 제외하지 않음) | 5,765 | - | - |
| 6-1 | 비교군: 저빈도 외식군 (L_OUT_FQ 5·6·7) | 3,062 | - | - |
| 6-2 | 중빈도 외식군 (L_OUT_FQ 4) | 797 | - | - |
| 6-3 | 노출군: 고빈도 외식군 (L_OUT_FQ 1·2·3) | 1,906 | - | - |


In [5]:
# 4) 설계 기반 추정 함수 svymean_domain (층 kstrata, 집락 psu, 가중치 WEIGHT; 도메인 분석)
# 설계 틀 F = 설계변수가 유효한 모든 행. 분석 대상은 '도메인'(불리언)으로만 지정 -> 층·집락 구조를 깨지 않음.
F   = df[df['kstrata'].notna() & df['psu_ok']].reset_index(drop=True)
cid = pd.factorize(F['kstrata'].astype(str) + '|' + F['psu'].astype(str))[0]     # (층, 집락) 번호
hid = pd.factorize(F['kstrata'])[0]                                              # 층 번호
cl_h = pd.Series(hid).groupby(cid).first().sort_index().to_numpy()               # 집락 -> 층
NCL  = len(cl_h)
Wt   = F['w'].fillna(0).to_numpy(float)
n_per_stratum = pd.Series(cl_h).value_counts()
N_STRATA = int(len(n_per_stratum)); N_LONELY = int((n_per_stratum == 1).sum())
print(f'설계 틀: {len(F):,}행, 층 {N_STRATA}개, 집락 {NCL}개, 집락이 1개뿐인 층 {N_LONELY}개')

def svymean_domain(y, domain):
    '''도메인 안의 가중 평균(0/1이면 가중 비율)과 선형화(테일러) 표준오차. 집락 1개 층은 분산 기여 0, FPC 무시.'''
    y = np.asarray(y, float)
    dm = np.asarray(domain, bool) & ~np.isnan(y) & (Wt > 0)
    wd = np.where(dm, Wt, 0.0)
    Wsum = wd.sum()
    if Wsum == 0: return np.nan, np.nan
    m = (wd * np.where(dm, y, 0.0)).sum() / Wsum
    u = wd * (np.where(dm, y, 0.0) - m) / Wsum
    tot = np.bincount(cid, weights=u, minlength=NCL)
    d = pd.DataFrame({'h': cl_h, 't': tot})
    g = d.groupby('h')['t']
    n_h, mean_h = g.count(), g.mean()
    ss = ((d['t'] - d['h'].map(mean_h)) ** 2).groupby(d['h']).sum()
    var = (n_h / (n_h - 1) * ss)[n_h > 1].sum()
    return float(m), float(np.sqrt(var))

def wskew(y, w):
    y, w = np.asarray(y, float), np.asarray(w, float)
    m = np.average(y, weights=w); s2 = np.average((y - m) ** 2, weights=w)
    return float(np.average((y - m) ** 3, weights=w) / s2 ** 1.5)

def wquantile(y, w, q):
    y, w = np.asarray(y, float), np.asarray(w, float)
    o = np.argsort(y); y, w = y[o], w[o]
    cw = np.cumsum(w) - 0.5 * w; cw /= w.sum()
    return float(np.interp(q, cw, y))

# 검증: 전체 도메인 평균 BMI 와 단순 가중 평균이 일치하는지
_A = F['analytic'].to_numpy()
_chk = np.average(F.loc[_A, 'bmi'], weights=F.loc[_A, 'w'])
assert abs(svymean_domain(F['bmi'].to_numpy(float), _A)[0] - _chk) < 1e-9
print('svymean_domain 평균 검증 통과')

설계 틀: 6,997행, 층 27개, 집락 192개, 집락이 1개뿐인 층 0개
svymean_domain 평균 검증 통과


## B-2 Table 1 / 비가중 표

In [6]:
import numpy as np
import pandas as pd
from IPython.display import display, HTML

# SMD 계산 함수 (연속형 및 비율/이분형 지원)
def calc_smd(m1, sd1, m2, sd2):
    pooled_sd = np.sqrt((sd1**2 + sd2**2) / 2)
    if pooled_sd == 0 or np.isnan(pooled_sd):
        return np.nan
    return (m2 - m1) / pooled_sd

COLS  = ['전체', '저빈도 외식군 (비교군)', '중빈도 외식군', '고빈도 외식군 (노출군)']
GVAL  = [None, 1, 2, 3]
ANAL  = F['analytic'].to_numpy()
DOM   = {c: ANAL & (np.ones(len(F), bool) if g is None else (F['grp'] == g).to_numpy()) for c, g in zip(COLS, GVAL)}
NHEAD = {c: int(DOM[c].sum()) for c in COLS}
IND   = '&nbsp;&nbsp;&nbsp;&nbsp;'  # HTML 들여쓰기 공백

GRP_NAME_MAP = {
    1: '저빈도 외식군 (비교군)',
    2: '중빈도 외식군',
    3: '고빈도 외식군 (노출군)'
}

def build_table(weighted):
    rows, stats = [], {}

    # 1. 외식 빈도 분포
    rows.append(['<b>외식 빈도 분포 (전체 열 기준)</b>', '', '', '', '', ''])
    for g in (1, 2, 3):
        name = GRP_NAME_MAP[g]
        n = NHEAD[name]
        p = svymean_domain((F['grp'] == g).to_numpy(float), DOM['전체'])[0] if weighted else n / NHEAD['전체']
        rows.append([IND + name, f'{n:,} ({p*100:.1f})', '', '', '', ''])

    # 2. 결과 (Outcome)
    rows.append(['<b>결과</b>', '', '', '', '', ''])
    for nm, var in [('BMI (kg/m²)', 'bmi')]:
        vals = []
        m_list, sd_list = {}, {}

        for c in COLS:
            y = F[var].to_numpy(float)
            if weighted:
                m, se = svymean_domain(y, DOM[c])
                yy = y[DOM[c] & ~np.isnan(y)]
                sd = yy.std(ddof=1) if len(yy) > 1 else 0
                vals.append(f'{m:.2f} ({se:.2f})')
            else:
                yy = y[DOM[c] & ~np.isnan(y)]
                m = yy.mean()
                sd = yy.std(ddof=1)
                vals.append(f'{m:.2f} ({sd:.2f})')

            m_list[c] = m
            sd_list[c] = sd
            stats[(nm, '평균', c)] = m

        smd_val = calc_smd(
            m_list['저빈도 외식군 (비교군)'], sd_list['저빈도 외식군 (비교군)'],
            m_list['고빈도 외식군 (노출군)'], sd_list['고빈도 외식군 (노출군)']
        )
        smd_str = f'{smd_val:.3f}' if not np.isnan(smd_val) else '-'
        rows.append([IND + nm + (', 가중 평균 (SE)' if weighted else ', 평균 (SD)')] + vals + [smd_str])

    # 비만 유병률
    vals = []
    p_list = {}
    for c in COLS:
        n = int((DOM[c] & (F['obese25'] == 1).to_numpy()).sum())
        p = svymean_domain(F['obese25'].to_numpy(float), DOM[c])[0] if weighted else n / NHEAD[c]
        vals.append(f'{n:,} ({p*100:.1f})')
        p_list[c] = p

    p1 = p_list['저빈도 외식군 (비교군)']
    p2 = p_list['고빈도 외식군 (노출군)']
    smd_obese = calc_smd(p1, np.sqrt(p1 * (1 - p1)), p2, np.sqrt(p2 * (1 - p2)))
    smd_obese_str = f'{smd_obese:.3f}' if not np.isnan(smd_obese) else '-'
    rows.append([IND + '비만 유병(BMI ≥ 25 kg/m²), n (%)'] + vals + [smd_obese_str])

    # 3. 공변량
    rows.append(['<b>공변량</b>', '', '', '', '', ''])

    # 연령
    vals, miss = [], []
    m_list, sd_list = {}, {}
    for c in COLS:
        y = F['age_n'].to_numpy(float)
        if weighted:
            m, se = svymean_domain(y, DOM[c])
            yy = y[DOM[c] & ~np.isnan(y)]
            sd = yy.std(ddof=1) if len(yy) > 1 else 0
            vals.append(f'{m:.2f} ({se:.2f})')
        else:
            yy = y[DOM[c] & ~np.isnan(y)]
            m = yy.mean()
            sd = yy.std(ddof=1)
            vals.append(f'{m:.2f} ({sd:.2f})')

        m_list[c] = m
        sd_list[c] = sd
        stats[('연령(세)', '평균', c)] = m
        miss.append(f"{int((DOM[c] & np.isnan(y)).sum()):,}")

    smd_age = calc_smd(
        m_list['저빈도 외식군 (비교군)'], sd_list['저빈도 외식군 (비교군)'],
        m_list['고빈도 외식군 (노출군)'], sd_list['고빈도 외식군 (노출군)']
    )
    smd_age_str = f'{smd_age:.3f}' if not np.isnan(smd_age) else '-'

    rows.append([IND + '연령 (세)' + (', 가중 평균 (SE)' if weighted else ', 평균 (SD)')] + vals + [smd_age_str])
    rows.append([IND + '연령 결측 n'] + miss + ['-'])

    # 다범주형 공변량
    sums = []
    for label, var, mapping, bad in CAT_SPECS:
        col = F['c_' + var]
        valid = col.notna().to_numpy()
        levels = list(dict.fromkeys(mapping.values()))
        rows.append([f'<b>{label}</b>', '', '', '', '', ''])
        tot = {c: 0.0 for c in COLS}

        for lv in levels:
            vals = []
            is_lv = (col == lv).to_numpy()
            p_list_lv = {}

            for c in COLS:
                n = int((DOM[c] & is_lv).sum())
                if weighted:
                    p = svymean_domain(is_lv.astype(float), DOM[c] & valid)[0]
                else:
                    nv = int((DOM[c] & valid).sum())
                    p = n / nv if nv else np.nan

                tot[c] += 0 if np.isnan(p) else p * 100
                stats[(label, lv, c)] = (n, p * 100)
                vals.append(f'{n:,} ({p*100:.1f})')
                p_list_lv[c] = p

            p1_lv = p_list_lv['저빈도 외식군 (비교군)']
            p2_lv = p_list_lv['고빈도 외식군 (노출군)']
            smd_lv = calc_smd(p1_lv, np.sqrt(p1_lv * (1 - p1_lv)), p2_lv, np.sqrt(p2_lv * (1 - p2_lv)))

            rows.append([IND + lv] + vals + [f'{smd_lv:.3f}' if not np.isnan(smd_lv) else '-'])

        rows.append([IND + '결측·무응답 n (비율의 분모에서 제외)'] +
                    [f"{int((DOM[c] & ~valid).sum()):,}" for c in COLS] + ['-'])
        sums.append((label, tot))

    hdr = ['변수'] + [f'{c}<br>(n={NHEAD[c]:,})' for c in COLS] + ['SMD<br>(비교 vs 노출)']
    return pd.DataFrame(rows, columns=hdr), stats, sums

# 1. 가중 표 생성
t_w, st_w, sums_w = build_table(True)

# 2. 코랩(Jupyter/Colab)에서 깔끔한 HTML 표 형태로 즉시 출력
print('=== Table 1 (가중) 결과 표 ===')
display(HTML(t_w.to_html(index=False, escape=False)))

=== Table 1 (가중) 결과 표 ===


변수,"전체(n=5,765)","저빈도 외식군 (비교군)(n=3,062)",중빈도 외식군(n=797),"고빈도 외식군 (노출군)(n=1,906)",SMD(비교 vs 노출)
외식 빈도 분포 (전체 열 기준),,,,,
저빈도 외식군 (비교군),"3,062 (46.0)",,,,
중빈도 외식군,797 (14.6),,,,
고빈도 외식군 (노출군),"1,906 (39.4)",,,,
결과,,,,,
"BMI (kg/m²), 가중 평균 (SE)",24.27 (0.07),23.97 (0.09),24.10 (0.19),24.67 (0.11),0.181
"비만 유병(BMI ≥ 25 kg/m²), n (%)","2,109 (37.8)","1,047 (34.3)",287 (36.9),775 (42.2),0.162
공변량,,,,,
"연령 (세), 가중 평균 (SE)",49.79 (0.49),57.13 (0.52),44.34 (0.79),43.23 (0.50),-0.920
연령 결측 n,0,0,0,0,-


#각주
1) n은 복합표본설계가 반영되지 않은 비가중 관측수이며, 백분율(%)과 평균은 복합표본가중치를 적용한 가중추정치임.

2) 연속형 변수(연령, BMI 등)는 분포의 정규성을 확인하기 위해 히스토그램과 왜도를 검토한 후 그 결과에 따라 요약통계량을 산출함.

3) 분석 시 복합표본설계요소(층화변수 [kstrata], 집락변수 [psu], 가중치 [wt_tot])를 반영하였으며, 전체 집단 내 특정 하위 그룹 추정을 위해 도메인 분석을 수행함.

4) 결측 및 무응답 건수는 별도로 표기하였으며, 비율 산출 시 해당 건수는 분모에서 제외하였음.

5) 본 분석에는 [국민건강영양조사 제9기 (2024년)] 원시자료가 사용되었음.

In [7]:
# 6) 연속형 요약통계 선택 근거: 가중 왜도
a = F[F['analytic']]
SKEW = {}
for nm, v in [('연령', 'age_n'), ('BMI', 'bmi')]:
    y = a[v].to_numpy(float); w = a['w'].to_numpy(float); ok = ~np.isnan(y)
    SKEW[nm] = wskew(y[ok], w[ok])
    q = [wquantile(y[ok], w[ok], p) for p in (0.25, 0.5, 0.75)]
    print(f'{nm}: 가중 왜도 {SKEW[nm]:.2f} | 가중 중앙값 {q[1]:.2f} (IQR {q[0]:.2f}-{q[2]:.2f})')
print('기준: |왜도| < 1 이면 대칭에 가까워 가중 평균 (SE)로 제시')

연령: 가중 왜도 0.02 | 가중 중앙값 50.00 (IQR 36.00-63.00)
BMI: 가중 왜도 0.86 | 가중 중앙값 23.84 (IQR 21.53-26.46)
기준: |왜도| < 1 이면 대칭에 가까워 가중 평균 (SE)로 제시


## B-3 비가중 대 가중

In [8]:
# 7) B-3: 비가중 표와 가중 표 비교 — 변수별 '표준화 차이(SMD)'가 큰 순서로 세 변수 선택
# 비교 기준(전체 열): 범주형은 각 범주의 (가중 % - 비가중 %) 와 SMD = 차이 / sqrt((p_w(1-p_w)+p_u(1-p_u))/2),
#                      연속형은 SMD = (가중 평균 - 비가중 평균) / 비가중 SD. 변수 점수 = 범주별 |SMD| 의 최댓값.
import numpy as np
import pandas as pd
from IPython.display import display, HTML

ANAL = F['analytic'].to_numpy()
IND = '&nbsp;&nbsp;&nbsp;&nbsp;'

def build_diff_comparison_table():
    table_rows = []

    # --- 1. 결과 (Outcome): BMI ---
    y_bmi = F['bmi'].to_numpy(float)
    yy_bmi = y_bmi[ANAL & ~np.isnan(y_bmi)]
    m_unw_bmi = yy_bmi.mean() if len(yy_bmi) > 0 else np.nan
    m_w_bmi, _ = svymean_domain(y_bmi, ANAL)
    diff_bmi = m_w_bmi - m_unw_bmi

    table_rows.append([
        IND + 'BMI (kg/m²), 평균',
        f'{m_unw_bmi:.2f}',
        f'{m_w_bmi:.2f}',
        f'{diff_bmi:+.2f}'
    ])

    # --- 2. 비만 유병률 ---
    valid_mask_ob = ANAL & F['obese25'].notna().to_numpy()
    nv_ob = int(valid_mask_ob.sum())
    n_unw_ob = int((valid_mask_ob & (F['obese25'] == 1).to_numpy()).sum())
    p_unw_ob = (n_unw_ob / nv_ob) * 100 if nv_ob > 0 else 0

    n_w_ob = int((ANAL & (F['obese25'] == 1).to_numpy()).sum())
    p_w_ob = svymean_domain(F['obese25'].to_numpy(float), ANAL)[0] * 100
    diff_ob = p_w_ob - p_unw_ob

    table_rows.append([
        IND + '비만 유병 (BMI ≥ 25), n (%)',
        f'{n_unw_ob:,} ({p_unw_ob:.1f}%)',
        f'{n_w_ob:,} ({p_w_ob:.1f}%)',
        f'{diff_ob:+.1f}%p'
    ])

    # --- 3. 공변량: 연령 ---
    table_rows.append(['<b>공변량</b>', '', '', ''])

    y_age = F['age_n'].to_numpy(float)
    yy_age = y_age[ANAL & ~np.isnan(y_age)]
    m_unw_age = yy_age.mean() if len(yy_age) > 0 else np.nan
    sd_unw_age = yy_age.std(ddof=1) if len(yy_age) > 1 else 0

    m_w_age, _ = svymean_domain(y_age, ANAL)
    diff_age = m_w_age - m_unw_age

    table_rows.append([
        IND + '연령 (세), 평균 (SD 또는 SE)',
        f'{m_unw_age:.1f} (SD:{sd_unw_age:.1f})',
        f'{m_w_age:.1f} (SE:{_:.2f})',
        f'{diff_age:+.1f}'
    ])

    miss_age = int((ANAL & np.isnan(y_age)).sum())
    table_rows.append([IND + '연령 결측 n', f'{miss_age:,}', f'{miss_age:,}', '-'])

    # --- 4. 다범주형 공변량 (CAT_SPECS 순회) ---
    for label, var, mapping, bad in CAT_SPECS:
        table_rows.append([f'<b>{label}</b>', '', '', ''])
        col = F['c_' + var]
        valid = col.notna().to_numpy()
        levels = list(dict.fromkeys(mapping.values()))

        for lv in levels:
            is_lv = (col == lv).to_numpy()

            # 비가중 비율
            nv = int((ANAL & valid).sum())
            n_unw = int((ANAL & is_lv).sum())
            p_unw = (n_unw / nv) * 100 if nv > 0 else 0

            # 가중 비율
            n_w = int((ANAL & is_lv).sum())
            p_w = svymean_domain(is_lv.astype(float), ANAL & valid)[0] * 100
            diff_p = p_w - p_unw

            table_rows.append([
                IND + lv,
                f'{n_unw:,} ({p_unw:.1f}%)',
                f'{n_w:,} ({p_w:.1f}%)',
                f'{diff_p:+.1f}%p'
            ])

        miss_cat = int((ANAL & ~valid).sum())
        table_rows.append([IND + '결측·무응답 n', f'{miss_cat:,}', f'{miss_cat:,}', '-'])

    columns = ['변수', '비가중 (Unweighted)', '가중 (Weighted)', '차이 (가중 - 비가중)']
    return pd.DataFrame(table_rows, columns=columns)

# 표 생성 및 코랩 HTML 출력
t_diff = build_diff_comparison_table()
print('=== 가중 vs 비가중 차이 비교 표 ===')
display(HTML(t_diff.to_html(index=False, escape=False)))

=== 가중 vs 비가중 차이 비교 표 ===


변수,비가중 (Unweighted),가중 (Weighted),차이 (가중 - 비가중)
"BMI (kg/m²), 평균",24.14,24.27,+0.13
"비만 유병 (BMI ≥ 25), n (%)","2,109 (36.6%)","2,109 (37.8%)",+1.2%p
공변량,,,
"연령 (세), 평균 (SD 또는 SE)",53.8 (SD:17.1),49.8 (SE:0.49),-4.1
연령 결측 n,0,0,-
성별,,,
남성,"2,474 (42.9%)","2,474 (50.0%)",+7.1%p
여성,"3,291 (57.1%)","3,291 (50.0%)",-7.1%p
결측·무응답 n,0,0,-
지역,,,
